In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# Device config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_epochs = 10
batch_size = 100
learning_rate = 0.001

train_dataset = torchvision.datasets.MNIST(
    root = './data',
    train = True,
    transform = transforms.ToTensor(),
    download = True
)

test_dataset = torchvision.datasets.MNIST(
    root = './data',
    train = False,
    transform = transforms.ToTensor(),
    download = True
)

train_dataloader = DataLoader(
    dataset=train_dataset,batch_size=batch_size, shuffle = True
)

test_dataloader = DataLoader(
    dataset=test_dataset, batch_size=batch_size, shuffle = False
)

class NeuralNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer1=nn.Linear(784,128)
    self.layer2=nn.Linear(128,64)
    self.layer3=nn.Linear(64,10)

  def forward(self,x):
    x=x.reshape(-1,784)
    x= self.layer1(x)
    x=torch.relu(x)
    x=self.layer2(x)
    x=torch.relu(x)
    x=self.layer3(x)
    return x

model=NeuralNet().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(), lr = learning_rate)

for epoch in range(num_epochs):
  for images, labels in train_dataloader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  print(f'Epoch: {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}')

n_correct = 0
n_total = 0

with torch.no_grad():
  for images ,labels in test_dataloader:
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs,1)
    n_correct+=(predicted==labels).sum().item()
    n_total+=labels.size(0)


acc = (n_correct/n_total)*100
print(f'Accuracy:{acc:.2f}%')



100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 443kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.12MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.2MB/s]


Epoch: 1/10, Loss: 0.0821
Epoch: 2/10, Loss: 0.0959
Epoch: 3/10, Loss: 0.0593
Epoch: 4/10, Loss: 0.0592
Epoch: 5/10, Loss: 0.1023
Epoch: 6/10, Loss: 0.1620
Epoch: 7/10, Loss: 0.1092
Epoch: 8/10, Loss: 0.0038
Epoch: 9/10, Loss: 0.0111
Epoch: 10/10, Loss: 0.0250
Accuracy:97.58%
